- https://github.com/verl-project/verl/blob/v0.6.1/verl/workers/sharding_manager/megatron_vllm.py#L93

In [ ]:
from verl.models.mcore.weight_converter import McoreToHFWeightConverterBase

- https://github.com/verl-project/verl/blob/v0.6.1/verl/models/mcore/weight_converter.py

In [ ]:
def __init__(
        self,
        actor_module: nn.ModuleList,
        inference_engine: LLM,
        model_config: DictConfig,
        transformer_config,
        rollout_config: DictConfig,
        layer_name_mapping,
        weight_converter: McoreToHFWeightConverterBase,
        device_mesh,
        offload_param: bool = True,
        bridge=None,
    ):

In [ ]:
# 调用权重转换器: Megatron 格式 → vLLM 格式
converted_names, converted_params = weight_converter.convert_param(cur_name, infer_params)


In [ ]:
class McoreToHFWeightConverterBase:
    def __init__(self, hf_config: PretrainedConfig, mcore_config: TransformerConfig):
        self.hf_config = hf_config
        self.mcore_config = mcore_config

    def convert_param(self, name: str, params_one_group: list[torch.Tensor]) -> torch.Tensor:
        raise NotImplementedError


- class McoreToHFWeightConverterDense(McoreToHFWeightConverterBase):
- class McoreToHFWeightConverterQwen2Moe(McoreToHFWeightConverterDense):
- class McoreToHFWeightConverterQwen2_5_VL(McoreToHFWeightConverterDense):
- class McoreToHFWeightConverterMixtral(McoreToHFWeightConverterDense):
- class McoreToHFWeightConverterQwen3Moe(McoreToHFWeightConverterDense):

# weight_converter.convert_param
- 这是 Megatron-LM 与 vLLM 之间的 权重格式转换 核心逻辑

## 为什么需要转换？

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│                    两种框架的权重布局差异                      │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Megatron-LM (NVIDIA)                                        │
│  ├── Tensor Parallel (TP): 按列/行切分 Linear 权重           │
│  │   ColumnParallelLinear:  weight = [in_dim, out_dim/tp]    │
│  │   RowParallelLinear:     weight = [in_dim/tp, out_dim]   │
│  ├── Pipeline Parallel (PP): 不同 stage 存不同层              │
│  └── 保存格式: 每个 tp/pp rank 一个 checkpoint 文件            │
│      model_optim_rng.pt (tp00-pp00, tp01-pp00, ...)         │
│                                                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  vLLM                                                        │
│  ├── 单卡推理，需要完整（非切分）权重                          │
│  ├── 布局: HuggingFace 风格或自定义格式                        │
│  │   Linear: weight = [out_dim, in_dim]  (PyTorch 默认)     │
│  └── 期望: 一个完整的 .bin/.safetensors 文件，或按层分布       │
│                                                             │
└─────────────────────────────────────────────────────────────┘

核心差异：
1. 切分 vs 完整: Megatron TP 切分 → vLLM 要合并
2. 转置: Megatron [in, out] → vLLM [out, in] (PyTorch Linear)
3. 命名: megatron.layers.0.attention.qkv → model.layers.0.self_attn.qkv
4. 特殊结构: Megatron 的 fused qkv, gated mlp 等 → vLLM 的独立投影

## convert_param 的核心功能

In [ ]:
class WeightConverter:
    """
    Megatron-LM checkpoint → vLLM 可用格式
    """
    
    def convert_param(self, megatron_param, param_name, config):
        """
        单参数转换入口
        
        Args:
            megatron_param: 从 Megatron checkpoint 加载的张量
            param_name: 参数在 Megatron 中的完整路径名
                       例: "language_model.encoder.layers.0.self_attention.query_key_value.weight"
            config: 模型配置 (hidden_size, num_heads, num_experts 等)
        
        Returns:
            vllm_param: 转换后的张量，可直接加载到 vLLM
            vllm_name: vLLM 中的参数名
        """
        # 1. 解析参数类型和位置
        param_info = self._parse_megatron_name(param_name)
        
        # 2. 处理 TP 切分（如果是切分状态）
        if param_info.is_tp_sharded:
            megatron_param = self._gather_tp_shards(megatron_param, param_info)
        
        # 3. 执行具体转换（转置、拆分、重命名等）
        converted = self._transform_param(megatron_param, param_info, config)
        
        # 4. 生成 vLLM 参数名
        vllm_name = self._map_to_vllm_name(param_name, param_info)
        
        return converted, vllm_name

## 关键转换场景详解
### 场景 1: Attention QKV 融合 → 拆分


In [ ]:
Megatron 格式 (TP=2):
┌─────────────────────────────────────────┐
│  TP-rank-0:                             │
│    qkv_weight = [hidden_dim, 3*hidden_dim/tp]  │
│    = [4096, 6144]  (concat q|k|v)       │
│                                         │
│  TP-rank-1:                             │
│    qkv_weight = [4096, 6144]             │
│                                         │
│  合并后: [4096, 12288] = [4096, 3*4096]   │
└─────────────────────────────────────────┘

vLLM 期望:
┌─────────────────────────────────────────┐
│  q_proj.weight = [4096, 4096]           │
│  k_proj.weight = [4096, 4096]           │
│  v_proj.weight = [4096, 4096]           │
│  或合并: qkv_proj.weight = [12288, 4096]  │
│  注意转置！                              │
└─────────────────────────────────────────┘

In [ ]:
def _transform_qkv(self, megatron_qkv, param_info, config):
    """
    Megatron fused qkv → vLLM q/k/v 投影
    """
    hidden_size = config.hidden_size
    num_heads = config.num_attention_heads
    num_kv_heads = getattr(config, 'num_key_value_heads', num_heads)
    head_dim = hidden_size // num_heads
    
    # Step 1: 如果是 TP 切分，先 all-gather
    if param_info.tp_sharded:
        # Column parallel: 按输出维度切分
        # megatron_qkv 形状: [hidden_size, 3*hidden_size/tp]
        full_qkv = torch.cat(megatron_qkv, dim=1)  # [hidden_size, 3*hidden_size]
    else:
        full_qkv = megatron_qkv
    
    # Step 2: Megatron 是 [in, out]，转置为 [out, in] (PyTorch Linear 标准)
    full_qkv = full_qkv.t()  # [3*hidden_size, hidden_size]
    
    # Step 3: 拆分为 q, k, v
    # Megatron 布局: [q_heads*head_dim | k_heads*head_dim | v_heads*head_dim]
    q_size = num_heads * head_dim
    kv_size = num_kv_heads * head_dim
    
    q = full_qkv[0:q_size]                          # [q_size, hidden_size]
    k = full_qkv[q_size:q_size+kv_size]             # [kv_size, hidden_size]
    v = full_qkv[q_size+kv_size:q_size+2*kv_size]   # [kv_size, hidden_size]
    
    # Step 4: GQA (Grouped Query Attention) 处理
    if num_kv_heads != num_heads:
        # k, v 需要重复以匹配 q 的头数
        k = self._repeat_kv_heads(k, num_heads, num_kv_heads)
        v = self._repeat_kv_heads(v, num_heads, num_kv_heads)
    
    return {
        'q_proj.weight': q,      # [hidden_size, hidden_size] 或调整后的形状
        'k_proj.weight': k,
        'v_proj.weight': v,
    }

### 场景 2: MLP (Gate-Up-Down) 转换

In [ ]:
Megatron 格式 (SwiGLU):
┌─────────────────────────────────────────┐
│  ColumnParallelLinear:                  │
│    gate_proj = [hidden_size, ffn_dim/tp] │
│    up_proj   = [hidden_size, ffn_dim/tp] │
│    (Megatron 通常 fuse 为 gate_up_proj)  │
│                                         │
│  RowParallelLinear:                     │
│    down_proj = [ffn_dim/tp, hidden_size] │
└─────────────────────────────────────────┘

vLLM 期望:
┌─────────────────────────────────────────┐
│  gate_proj.weight = [ffn_dim, hidden_size] │
│  up_proj.weight   = [ffn_dim, hidden_size] │
│  down_proj.weight = [hidden_size, ffn_dim]  │
└─────────────────────────────────────────┘

In [ ]:
def _transform_mlp(self, megatron_mlp, param_info, config):
    """
    Megatron MLP → vLLM MLP
    """
    ffn_dim = config.intermediate_size
    
    if 'gate_up_proj' in param_info.name:
        # Megatron fuse 了 gate 和 up
        # shape: [hidden_size, 2*ffn_dim/tp] (column parallel)
        
        full_gate_up = torch.cat(megatron_mlp, dim=1)  # [hidden_size, 2*ffn_dim]
        full_gate_up = full_gate_up.t()                 # [2*ffn_dim, hidden_size]
        
        gate = full_gate_up[0:ffn_dim]      # [ffn_dim, hidden_size]
        up = full_gate_up[ffn_dim:2*ffn_dim] # [ffn_dim, hidden_size]
        
        return {
            'gate_proj.weight': gate,
            'up_proj.weight': up,
        }
    
    elif 'down_proj' in param_info.name:
        # Row parallel: [ffn_dim/tp, hidden_size]
        full_down = torch.cat(megatron_mlp, dim=0)  # [ffn_dim, hidden_size]
        # 已经是 [out, in] 格式，无需转置
        return {'down_proj.weight': full_down}

### 场景 3: MoE Expert 参数转换

In [ ]:
Megatron MoE 格式 (EP=4, TP=2):
┌─────────────────────────────────────────────────────────┐
│  EP-rank-0:                                             │
│    experts.0.w1 = [hidden_size, ffn_dim/tp]  (TP-0)    │
│    experts.0.w2 = [ffn_dim/tp, hidden_size]    (TP-0)    │
│    experts.1.w1 = ...                                   │
│    ...                                                  │
│  EP-rank-1:                                             │
│    experts.2.w1 = ...                                   │
│    experts.3.w1 = ...                                   │
│  ...                                                    │
└─────────────────────────────────────────────────────────┘

vLLM 期望:
┌─────────────────────────────────────────────────────────┐
│  model.layers.0.mlp.experts.0.gate_proj.weight         │
│  model.layers.0.mlp.experts.0.up_proj.weight           │
│  model.layers.0.mlp.experts.0.down_proj.weight           │
│  ...                                                    │
│  model.layers.0.mlp.experts.7.gate_proj.weight           │
└─────────────────────────────────────────────────────────┘

In [ ]:
def _transform_moe_expert(self, megatron_expert, param_info, config):
    """
    Megatron MoE Expert → vLLM Expert
    需要跨 EP 收集所有 expert
    """
    num_experts = config.num_experts
    ep_size = config.ep_size
    
    # Step 1: 收集所有 EP rank 的 expert
    # 当前只有本地 EP rank 的 expert（如 expert 0,1）
    all_experts = self._gather_ep_experts(megatron_expert, ep_size)
    # all_experts[i] = expert i 的参数
    
    results = {}
    for expert_id in range(num_experts):
        expert_param = all_experts[expert_id]
        
        # 每个 expert 内部可能还有 TP 切分
        if param_info.tp_sharded:
            expert_param = torch.cat(expert_param, dim=1 if 'w1' in param_info.name else 0)
        
        # 转置和重命名
        if 'w1' in param_info.name:  # gate/up
            expert_param = expert_param.t()
            # 拆分为 gate 和 up...
            results[f'experts.{expert_id}.gate_proj.weight'] = gate
            results[f'experts.{expert_id}.up_proj.weight'] = up
        elif 'w2' in param_info.name:  # down
            # 已经是 [out, in]，但可能需要调整
            results[f'experts.{expert_id}.down_proj.weight'] = expert_param
    
    return results

## 完整的 convert_param 实现


In [ ]:
class MegatronToVLLMConverter:
    def __init__(self, megatron_config, vllm_config):
        self.m_config = megatron_config
        self.v_config = vllm_config
        self.tp_size = megatron_config.tensor_model_parallel_size
        self.pp_size = megatron_config.pipeline_model_parallel_size
        self.ep_size = getattr(megatron_config, 'expert_model_parallel_size', 1)
    
    def convert_param(self, param_tensor, param_name, tp_rank=0, pp_rank=0, ep_rank=0):
        """
        主转换函数
        
        Args:
            param_tensor: 从 Megatron checkpoint 读出的张量
            param_name: 如 "language_model.encoder.layers.0.self_attention.query_key_value.weight"
            tp_rank: 当前张量来自哪个 TP rank
            pp_rank: 当前张量来自哪个 PP rank  
            ep_rank: 当前张量来自哪个 EP rank (MoE)
        """
        # 1. 解析参数名，识别类型和位置
        parsed = self._parse_name(param_name)
        
        # 2. 如果是 PP 切分，需要按层号路由到对应位置
        if self.pp_size > 1:
            layer_offset = pp_rank * self._layers_per_pp_stage()
            parsed.layer_idx += layer_offset
        
        # 3. 收集 TP 切分的碎片（如果需要）
        if parsed.is_tp_sharded and self.tp_size > 1:
            # 实际实现中，这里会从所有 TP rank 读取并合并
            param_tensor = self._gather_tp_shard(param_tensor, parsed, tp_rank)
        
        # 4. 收集 EP 切分的 expert（如果是 MoE）
        if parsed.is_expert and self.ep_size > 1:
            param_tensor = self._gather_ep_expert(param_tensor, parsed, ep_rank)
        
        # 5. 执行具体转换
        converted = self._apply_transform(param_tensor, parsed)
        
        # 6. 生成 vLLM 参数名
        vllm_name = self._to_vllm_name(parsed)
        
        return converted, vllm_name
    
    def _parse_name(self, name):
        """解析 Megatron 参数名"""
        parts = name.split('.')
        
        info = ParamInfo()
        
        # 识别层号
        if 'layers' in parts:
            idx = parts.index('layers')
            info.layer_idx = int(parts[idx + 1])
        
        # 识别参数类型
        if 'self_attention' in name or 'attention' in name:
            info.module = 'attention'
            if 'query_key_value' in name or 'qkv' in name:
                info.param_type = 'qkv_proj'
                info.is_tp_sharded = True  # column parallel
            elif 'dense' in name or 'o_proj' in name:
                info.param_type = 'o_proj'
                info.is_tp_sharded = True  # row parallel
        elif 'mlp' in name or 'feed_forward' in name:
            info.module = 'mlp'
            if 'gate' in name and 'up' in name:
                info.param_type = 'gate_up_proj'
                info.is_tp_sharded = True
            elif 'down' in name or 'wo' in name:
                info.param_type = 'down_proj'
                info.is_tp_sharded = True
        elif 'experts' in name or 'expert' in name:
            info.module = 'moe'
            info.is_expert = True
            # 提取 expert_id
            if 'experts' in parts:
                idx = parts.index('experts')
                info.expert_id = int(parts[idx + 1])
        
        return info
    
    def _gather_tp_shard(self, tensor, info, tp_rank):
        """合并 TP 切分的张量"""
        if info.param_type in ['qkv_proj', 'gate_up_proj', 'gate_proj', 'up_proj']:
            # Column parallel: 按输出维度切分，需要 cat(dim=1)
            # 实际实现需要从所有 TP rank 读取
            shards = self._load_all_tp_shards(info)
            return torch.cat(shards, dim=1)
        elif info.param_type in ['o_proj', 'down_proj']:
            # Row parallel: 按输入维度切分，需要 cat(dim=0)
            shards = self._load_all_tp_shards(info)
            return torch.cat(shards, dim=0)
        return tensor
    
    def _apply_transform(self, tensor, info):
        """执行具体的张量变换"""
        # Megatron 默认布局: [in_features, out_features]
        # PyTorch/vLLM Linear: [out_features, in_features]
        needs_transpose = True
        
        if info.module == 'attention':
            if info.param_type == 'qkv_proj':
                # [hidden_size, 3*hidden_size] → 需要拆分和转置
                tensor = tensor.t()  # [3*hidden_size, hidden_size]
                # 后续拆分为 q, k, v...
                return self._split_qkv(tensor)
            elif info.param_type == 'o_proj':
                # [hidden_size/tp, hidden_size] → cat 后 [hidden_size, hidden_size]
                tensor = tensor.t()  # [hidden_size, hidden_size]
        
        elif info.module == 'mlp':
            if info.param_type == 'gate_up_proj':
                tensor = tensor.t()
                return self._split_gate_up(tensor)
            elif info.param_type == 'down_proj':
                tensor = tensor.t()
        
        elif info.module == 'moe':
            # Expert 参数转换类似 MLP，但带 expert_id
            pass
        
        # 默认转置
        if needs_transpose:
            tensor = tensor.t()
        
        return tensor
    
    def _to_vllm_name(self, info):
        """生成 vLLM 参数名"""
        layer = f"model.layers.{info.layer_idx}"
        
        mapping = {
            'attention': {
                'qkv_proj': f"{layer}.self_attn.qkv_proj.weight",
                'o_proj': f"{layer}.self_attn.o_proj.weight",
            },
            'mlp': {
                'gate_proj': f"{layer}.mlp.gate_proj.weight",
                'up_proj': f"{layer}.mlp.up_proj.weight",
                'down_proj': f"{layer}.mlp.down_proj.weight",
            },
            'moe': {
                'gate_proj': f"{layer}.block_sparse_moe.experts.{info.expert_id}.w1.weight",
                'up_proj': f"{layer}.block_sparse_moe.experts.{info.expert_id}.w3.weight",
                'down_proj': f"{layer}.block_sparse_moe.experts.{info.expert_id}.w2.weight",
            }
        }
        
        return mapping[info.module][info.param_type]

## 使用流程

In [ ]:
# 1. 加载 Megatron checkpoint
from megatron.checkpointing import load_checkpoint

megatron_ckpt = load_checkpoint(
    checkpoint_path="/path/to/megatron/ckpt",
    tp_size=4,
    pp_size=2,
    ep_size=2,
)

# 2. 创建转换器
converter = MegatronToVLLMConverter(
    megatron_config=megatron_ckpt.config,
    vllm_config=vllm_model.config,
)

# 3. 遍历所有参数并转换
vllm_state_dict = {}
for (name, tensor, tp_rank, pp_rank, ep_rank) in megatron_ckpt.iter_params():
    vllm_tensor, vllm_name = converter.convert_param(
        tensor, name, tp_rank, pp_rank, ep_rank
    )
    vllm_state_dict[vllm_name] = vllm_tensor

# 4. 保存为 vLLM 格式
torch.save(vllm_state_dict, "vllm_model.pt")

# 5. 或直接加载到 vLLM
vllm_model.load_weights(vllm_state_dict)

## 关键难点总结

| 难点                  | 解决方案                             |
| ------------------- | -------------------------------- |
| TP 切分合并             | 识别 column/row parallel，正确 cat    |
| 命名映射                | 建立完整的 name mapping 表             |
| QKV 融合拆分            | 按 head 维度拆分，处理 GQA               |
| SwiGLU 格式           | gate/up 融合 → 拆分                  |
| MoE Expert 收集       | 跨 EP all-gather，再按 expert\_id 分发 |
| 精度对齐                | 确保 bf16/fp16 转换一致，验证输出           |
| LayerNorm/Embedding | 通常不切分，直接复制                       |
